# 02. Temporal Feature Engineering Pipeline
### Industrial Predictive Maintenance Portfolio Project | NASA C-MAPSS Dataset

---

## 1. Motivation & Technical Objectives
Raw snapshot sensor telemetry suffers from high-frequency measurement jitter and does not inherently reflect the temporal history of degradation. To provide regressors with degradation context, we engineer three classes of features:

1. **Engine-Isolated Rolling Window Statistics** ($\mu, \sigma, \min, \max$ over window $w=10$): Captures trend and operational volatility without crossing engine boundaries.
2. **Cumulative Absolute Change (Wear Proxy)**: Monotonically proxies accumulated thermal and mechanical wear.
3. **Normalized Cycle Position**: Represents relative engine aging within observed telemetry.
4. **Piecewise Linear RUL Clipping ($RUL_{clip} = 125$)**: Capping the target RUL focuses model capacity exclusively on the active degradation zone.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_dataset, SENSOR_COLUMNS
from src.preprocessor import add_rul_labels, FD001_LOW_VARIANCE_SENSORS, RUL_CLIP
from src.feature_engineer import build_feature_pipeline, add_rolling_features, add_cumulative_wear_features

## 2. Piecewise Linear RUL Target Formulation
In industrial turbofans, an engine at cycle 5 with 300 cycles to failure exhibits identical telemetry to cycle 50 with 255 cycles to failure. Early in lifecycle, no wear is detectable. We cap $RUL \le 125$.

In [ ]:
df_raw = load_dataset('FD001', 'train')
df_unclipped = add_rul_labels(df_raw, clip_rul=False)
df_clipped = add_rul_labels(df_raw, clip_rul=True, rul_clip_limit=RUL_CLIP)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_unclipped[df_unclipped['engine_id'] == 1]['cycle'], df_unclipped[df_unclipped['engine_id'] == 1]['RUL'], label='Linear RUL (Unclipped)', color='#e74c3c', linestyle='--')
ax.plot(df_clipped[df_clipped['engine_id'] == 1]['cycle'], df_clipped[df_clipped['engine_id'] == 1]['RUL'], label=f'Piecewise Linear RUL (Clipped at {RUL_CLIP})', color='#2ecc71', linewidth=2)
ax.set_xlabel('Operational Flight Cycle')
ax.set_ylabel('Target RUL Label')
ax.set_title('Piecewise Linear RUL Labeling Strategy (Engine 1)')
ax.legend()
plt.show()

## 3. Rolling Window Feature Extraction
Calculated strictly per engine using `groupby('engine_id').transform(...)` with `min_periods=1`.

In [ ]:
retained_sensors = [s for s in SENSOR_COLUMNS if s not in FD001_LOW_VARIANCE_SENSORS]
df_feat, _ = build_feature_pipeline(df_clipped, sensor_cols=retained_sensors, window=10)
print(f"Constructed feature matrix with {df_feat.shape[1]} columns across {df_feat.shape[0]} rows.")
df_feat[['engine_id', 'cycle', 's2', 's2_mean_10', 's2_std_10', 's2_cum_change', 'cycle_norm', 'RUL']].head(12)

## 4. Verification: No Data Leakage Across Engine Boundaries
Validating that rolling and cumulative features reset to 0 at cycle 1 of every engine.

In [ ]:
e1_end = df_feat[df_feat['engine_id'] == 1].iloc[-1]
e2_start = df_feat[df_feat['engine_id'] == 2].iloc[0]

print("Engine 1 Final Cycle Cumulative Change (s2):", e1_end['s2_cum_change'])
print("Engine 2 Cycle 1 Cumulative Change (s2)   :", e2_start['s2_cum_change'])
assert e2_start['s2_cum_change'] == 0.0, "Leakage detected across engine boundary!"
print("Assertion Passed: Engine-level isolation confirmed.")